In [32]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import itertools
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor
import numpy as np
from plotly.subplots import make_subplots

datos_unidades = pd.read_csv('../data_clean/datos_unidades.csv')
ventas_completo = pd.read_csv('../data_clean/ventas_clean.csv')
clientes = pd.read_csv('../data_raw/clientes.csv')


ventas_completo['Fecha'] = pd.to_datetime(ventas_completo['Fecha'])
datos_unidades['Fecha'] = pd.to_datetime(datos_unidades['Fecha'])

## Modelo

In [34]:
def entrenar_regresion(df):
    df_reg = df.copy()
    
    df_reg['Dias'] = (df_reg['Fecha'] - df_reg['Fecha'].min()).dt.days
    
    X = df_reg[['Dias']]
    y = df_reg['Unidades_Acumuladas']
    
    # Train / Test split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, shuffle=False
    )
    
    # Modelo
    modelo = LinearRegression()
    modelo.fit(X_train, y_train)
    
    # Predicción completa
    df_reg['Predicción'] = modelo.predict(X)
    
    # Metricas
    mse = mean_squared_error(y_test, modelo.predict(X_test))
    rmse = np.sqrt(mse)   #Calculoo RMSE
    
    r2 = r2_score(y_test, modelo.predict(X_test))
    
    return df_reg, rmse, r2, modelo

In [35]:
fig_unidades = px.line(
    datos_unidades, 
    x='Fecha', 
    y='Unidades_Acumuladas',
    title='Crecimiento de Volumen de Ventas (Unidades Acumuladas)',
    labels={'Unidades_Acumuladas': 'Total Unidades Vendidas (u)', 'Fecha': 'Fecha'},
    markers=True
)

fig_unidades.update_layout(
    hovermode="x unified",
    xaxis_title="Tiempo",
    yaxis_title="Unidades (u)"
)

fig_unidades.show()

# GRÁFICO 2: DESGLOSE POR CATEGORÍA 
# agrupar por Fecha y Categoría usando CANTIDAD
datos_cat_unidades = ventas_completo.groupby(['Fecha', 'Categoría'])[['Cantidad']].sum().reset_index()
datos_cat_unidades = datos_cat_unidades.sort_values('Fecha')

# calcular acumulado por grupo
datos_cat_unidades['Unidades_Acumuladas'] = datos_cat_unidades.groupby('Categoría')['Cantidad'].cumsum()
fig_cat_unidades = px.line(
    datos_cat_unidades, 
    x='Fecha', 
    y='Unidades_Acumuladas', 
    color='Categoría',
    title='Unidades Acumuladas por Categoría de Producto',
    labels={'Unidades_Acumuladas': 'Unidades Acumuladas (u)'}
)

fig_cat_unidades.show()

In [36]:
ventas_completo

,Unnamed: 0,ID_Venta,Fecha,ID_Cliente,ID_Producto,Cantidad,Método_Pago,Estado,Semana,Mes,Dia_Semana,Trimestre,Nombre_producto,Categoría,Precio_Unitario,Stock,Venta_Total
0,0,919,2024-01-31,10,25,5,1,Completa,5,1,Wednesday,1,Pizza congelada,Congelados,15.45,1640,77.25
1,1,947,2024-01-31,106,5,1,4,Completa,5,1,Wednesday,1,Manteca,Lácteos,5.65,4929,5.65
2,2,1317,2024-01-31,235,25,3,3,Completa,5,1,Wednesday,1,Pizza congelada,Congelados,15.45,1640,46.35
3,3,1607,2024-01-31,114,15,5,1,Completa,5,1,Wednesday,1,Medialunas,Panadería,3.51,4043,17.55
4,4,2038,2024-01-31,132,2,5,4,Completa,5,1,Wednesday,1,Yogur,Lácteos,5.21,3358,26.05
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2995,3024,954,2024-12-30,44,12,6,4,Completa,1,12,Monday,4,Pan francés,Panadería,8.12,1619,48.72
2996,3025,1390,2024-12-30,26,31,3,4,Completa,1,12,Monday,4,Galletitas de agua,Galletitas y Snacks,5.24,2771,15.72
2997,3026,1519,2024-12-30,246,11,3,3,Completa,1,12,Monday,4,Salchicha,Carnicería,11.23,1726,33.69
2998,3027,2147,2024-12-30,231,22,2,1,Pendiente,1,12,Monday,4,Cebolla,Frutas y Verduras,4.21,3545,8.42


In [37]:
df_total_pred, mse_total, r2_total, modelo_total = entrenar_regresion(datos_unidades)

print("RMSE total:", mse_total)
print("R2 total :", r2_total)

fig_reg = go.Figure()

fig_reg.add_trace(go.Scatter(
    x=df_total_pred['Fecha'], 
    y=df_total_pred['Unidades_Acumuladas'],
    mode='lines+markers',
    name='Ventas Acumuladas Reales'
))

fig_reg.add_trace(go.Scatter(
    x=df_total_pred['Fecha'], 
    y=df_total_pred['Predicción'],
    mode='lines',
    name='Regresión Lineal (Predicción)'
))

fig_reg.update_layout(
    title='Regresión Lineal sobre Ventas Acumuladas (Total)',
    xaxis_title='Fecha',
    yaxis_title='Unidades Acumuladas'
)

fig_reg.show()

RMSE total: 52.208781181232546
R2 total : 0.9920609011969701


In [38]:
metricas = []
df_predicciones = []

for cat, df_cat in datos_cat_unidades.groupby('Categoría'):
    
    df_pred, rmse, r2, modelo = entrenar_regresion(df_cat)  # ← rmse viene de la función
    df_pred['Categoría'] = cat
    
    df_predicciones.append(df_pred)
    metricas.append({'Categoría': cat, 'RMSE': rmse, 'R2': r2})  # ← guardar RMSE
      

df_reg_cat = pd.concat(df_predicciones, ignore_index=True)
df_metricas = pd.DataFrame(metricas)

print(df_metricas)

fig = px.line(
    df_reg_cat,
    x='Fecha',
    y='Unidades_Acumuladas',
    color='Categoría',
    title='Ventas Acumuladas Reales por Categoría'
)

fig_pred = px.line(
    df_reg_cat,
    x='Fecha',
    y='Predicción',
    color='Categoría',
    title='Predicción por Categoría (Regresión Lineal)',
    line_dash='Categoría'
)

fig.show()
fig_pred.show()

             Categoría       RMSE        R2
0              Bebidas  29.224829  0.837328
1           Carnicería  14.335241  0.974494
2           Congelados  26.648077  0.906910
3            Conservas  18.227941  0.914936
4    Frutas y Verduras  35.080863  0.802067
5  Galletitas y Snacks  32.052392  0.690701
6              Lácteos  46.623532  0.628150
7            Panadería  30.028081  0.864714


### Venta Proximos 20% de dias

In [39]:
# Gráficas
def plot_comparacion(df, titulo, marcar_corte=False):

    real_col = "Venta_Real" if "Venta_Real" in df.columns else "Cantidad"

    fig = make_subplots(rows=1, cols=1)

    fig.add_trace(go.Scatter(
        x=df["Fecha"], y=df[real_col],
        mode="lines", name="Real"
    ))

    fig.add_trace(go.Scatter(
        x=df["Fecha"], y=df["Prediccion"],
        mode="lines", name="Prediccion"
    ))

    if marcar_corte:
        f_corte = ventas_diarias_clean.iloc[corte]["Fecha"]
        fig.add_vline(x=f_corte)
        fig.add_annotation(x=f_corte, y=1, yref="paper",
                           text="Inicio Test", showarrow=False)

    fig.update_layout(
        title=titulo,
        xaxis_title="Fecha",
        yaxis_title="Unidades"
    )

    fig.show()


# Features
ventas_diarias = (
    ventas_completo
    .groupby("Fecha")["Cantidad"]
    .sum()
    .reset_index()
)

# Variables básicas
ventas_diarias["Dia_Semana"] = ventas_diarias["Fecha"].dt.dayofweek
ventas_diarias["Mes"] = ventas_diarias["Fecha"].dt.month

# Variables de dia específico 
ventas_diarias["EsLunes"] = (ventas_diarias["Dia_Semana"] == 0).astype(int)
ventas_diarias["EsMartes"] = (ventas_diarias["Dia_Semana"] == 1).astype(int)
ventas_diarias["EsMiercoles"] = (ventas_diarias["Dia_Semana"] == 2).astype(int)
ventas_diarias["EsJueves"] = (ventas_diarias["Dia_Semana"] == 3).astype(int)
ventas_diarias["EsViernes"] = (ventas_diarias["Dia_Semana"] == 4).astype(int)
ventas_diarias["EsSabado"] = (ventas_diarias["Dia_Semana"] == 5).astype(int)
ventas_diarias["EsDomingo"] = (ventas_diarias["Dia_Semana"] == 6).astype(int)

# Variables compuestas útiles
ventas_diarias["EsFinDeSemana"] = (ventas_diarias["Dia_Semana"] >= 5).astype(int)
ventas_diarias["EsInicioSemana"] = (ventas_diarias["Dia_Semana"] <= 1).astype(int)

# Variables de tendencia reciente
ventas_diarias["Lag1"] = ventas_diarias["Cantidad"].shift(1)  # Ayer
ventas_diarias["Lag2"] = ventas_diarias["Cantidad"].shift(2)  # Anteayer
ventas_diarias["Lag7"] = ventas_diarias["Cantidad"].shift(7)  # semana anterior

# Promedios móviles recientes
ventas_diarias["MA3"] = ventas_diarias["Cantidad"].rolling(3, min_periods=1).mean()  # Últimos 3 días
ventas_diarias["MA7"] = ventas_diarias["Cantidad"].rolling(7, min_periods=1).mean()  # Última semana
ventas_diarias["MA14"] = ventas_diarias["Cantidad"].rolling(14, min_periods=1).mean()  # Últimas 2 semanas

# Variables de estacionalidad mensual
ventas_diarias["EsInicioMes"] = (ventas_diarias["Fecha"].dt.day <= 7).astype(int)  # Primera semana
ventas_diarias["EsFinMes"] = (ventas_diarias["Fecha"].dt.day >= 25).astype(int)  # Última semana

# Features finales
features = [
    # Tendencias recientes
    "Lag1", "Lag2", "Lag7",
    "MA3", "MA7", "MA14",
    # Días específicos
   "EsMartes", "EsFinDeSemana","EsJueves","EsInicioSemana",
    # Patrones mensuales
    "EsFinMes"
]

# Limpiar solo las filas donde faltan lags (que son los primeros días)
ventas_diarias_clean = ventas_diarias.dropna(subset=["Lag1", "Lag2", "Lag7"]).reset_index(drop=True)

X = ventas_diarias_clean[features]
Y = ventas_diarias_clean["Cantidad"]

# División train/test 
corte = int(len(X) * 0.8)  # 80% entrenamiento, 20% test
X_train, X_test = X[:corte], X[corte:]
Y_train, Y_test = Y[:corte], Y[corte:]


# Entrenar y evaluar
def entrenar_y_evaluar(modelo, nombre_modelo):
    modelo.fit(X_train, Y_train)
    
    pred_train = modelo.predict(X_train)
    pred_test = modelo.predict(X_test)
    
    rmse_train = np.sqrt(mean_squared_error(Y_train, pred_train))
    rmse_test = np.sqrt(mean_squared_error(Y_test, pred_test))
    r2_train = r2_score(Y_train, pred_train)
    r2_test = r2_score(Y_test, pred_test)
    
    print(f"\n{nombre_modelo}")
    print(f"Train RMSE: {rmse_train:.2f}, R2: {r2_train:.4f}")
    print(f"Test RMSE: {rmse_test:.2f}, R2: {r2_test:.4f}")
    
    # DF con predicciones
    df_full = ventas_diarias_clean.copy()
    df_full["Prediccion"] = np.nan
    train_idx = X_train.index
    test_idx = X_test.index
    df_full.loc[train_idx, "Prediccion"] = pred_train
    df_full.loc[test_idx, "Prediccion"] = pred_test
    
    # DF de test
    df_test = ventas_diarias_clean.iloc[test_idx].copy()
    df_test["Venta_Real"] = Y_test
    df_test["Prediccion"] = pred_test
    
    return df_full, df_test

#Random Forest
modelo_rf = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1,
    max_depth=10
)

df_full_rf, df_test_rf = entrenar_y_evaluar(modelo_rf, "Random Forest")
plot_comparacion(df_full_rf, "Random Forest - Predicción Completa", marcar_corte=True)
plot_comparacion(df_test_rf, "Random Forest - Período de Test")

# Importancia de features
importancias = modelo_rf.feature_importances_
feature_importance_df = pd.DataFrame({
    'Feature': features,
    'Importance': importancias
}).sort_values('Importance', ascending=False)

print("Importancia de Features (en Random Forest):")
print(feature_importance_df)


Random Forest
Train RMSE: 2.24, R2: 0.9645
Test RMSE: 6.11, R2: 0.7011


Importancia de Features (en Random Forest):
           Feature  Importance
3              MA3    0.481820
1             Lag2    0.212521
0             Lag1    0.167335
4              MA7    0.043236
2             Lag7    0.040469
5             MA14    0.030830
7    EsFinDeSemana    0.005352
10        EsFinMes    0.005244
9   EsInicioSemana    0.004833
6         EsMartes    0.004494
8         EsJueves    0.003866


## Cancelación

In [40]:
Df_cancelaciones = ventas_completo.copy()
Df_cancelaciones = Df_cancelaciones[Df_cancelaciones["Estado"] != "Pendiente"]
mapeo = {"Completa": 0, "Cancelada": 1}
Df_cancelaciones["Cancelados"] = Df_cancelaciones["Estado"].map(mapeo)
mapeo_dias = {
    'Monday': 1,
    'Tuesday': 2,
    'Wednesday': 3,
    'Thursday': 4,
    'Friday': 5,
    'Saturday': 6,
    'Sunday': 7
}
Df_cancelaciones['Dia_Numerico'] = Df_cancelaciones['Dia_Semana'].map(mapeo_dias)
Df_cancelaciones = Df_cancelaciones.merge(clientes, on='ID_Cliente')
mapeo_regiones = {
    'Buenos Aires': 1,
    'Patagonia': 2,
    'Centro': 3,
    'Cuyo': 4,
    'NEA': 5,
    'NOA': 6
}
Df_cancelaciones['Region_Numerica'] = Df_cancelaciones['Región'].map(mapeo_regiones)

columnas_finales = [
    "ID_Cliente",
    "Cancelados", # Variable Objetivo
    "Método_Pago",
    "Venta_Total", 
    "Precio_Unitario",
    "Cantidad",
    "Dia_Numerico",
    "Mes",
    "Region_Numerica"
    ]

Df_modelado = Df_cancelaciones[columnas_finales]
Df_modelado

,ID_Cliente,Cancelados,Método_Pago,Venta_Total,Precio_Unitario,Cantidad,Dia_Numerico,Mes,Region_Numerica
0,10,0,1,77.25,15.45,5,3,1,1
1,106,0,4,5.65,5.65,1,3,1,5
2,235,0,3,46.35,15.45,3,3,1,5
3,114,0,1,17.55,3.51,5,3,1,3
4,132,0,4,26.05,5.21,5,3,1,1
...,...,...,...,...,...,...,...,...,...
2528,248,0,1,71.25,14.25,5,1,12,3
2529,44,0,4,48.72,8.12,6,1,12,1
2530,26,0,4,15.72,5.24,3,1,12,3
2531,246,0,3,33.69,11.23,3,1,12,5


In [41]:
ventas_completo.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 17 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   Unnamed: 0       3000 non-null   int64         
 1   ID_Venta         3000 non-null   int64         
 2   Fecha            3000 non-null   datetime64[ns]
 3   ID_Cliente       3000 non-null   int64         
 4   ID_Producto      3000 non-null   int64         
 5   Cantidad         3000 non-null   int64         
 6   Método_Pago      3000 non-null   int64         
 7   Estado           3000 non-null   object        
 8   Semana           3000 non-null   int64         
 9   Mes              3000 non-null   int64         
 10  Dia_Semana       3000 non-null   object        
 11  Trimestre        3000 non-null   int64         
 12  Nombre_producto  3000 non-null   object        
 13  Categoría        3000 non-null   object        
 14  Precio_Unitario  3000 non-null   float64